# Finite-Difference Playground

This notebook isolates one generated finite-difference idea: a fourth-order centered second derivative. It inspects the NRPy stencil coefficients, emits a small C expression, and builds a temporary C program when local build tools are available.

## Table of Contents

1. [Stencil goal](#Stencil-goal)
2. [NRPy state hygiene and imports](#NRPy-state-hygiene-and-imports)
3. [Finite-difference coefficients](#Finite-difference-coefficients)
4. [Small C expression](#Small-C-expression)
5. [Temporary C program](#Temporary-C-program)
6. [Build and run](#Build-and-run)
7. [Next notebooks](#Next-notebooks)

## Stencil Goal

For a uniform grid spacing $\Delta x$, the fourth-order centered approximation to $\partial_x^2\phi$ uses two points on each side of the target point. NRPy returns the coefficient list and stencil offsets from the derivative label `dDD00`.

## NRPy State Hygiene and Imports

The following cell clears only names introduced by this notebook, then registers two tutorial gridfunctions. This keeps reruns in the same kernel predictable without resetting unrelated NRPy state.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

import sympy as sp

import nrpy
import nrpy.c_codegen as ccg
import nrpy.c_function as cfc
import nrpy.finite_difference as fin
import nrpy.grid as gri
import nrpy.params as par


print("Imported nrpy from:", nrpy.__file__)

print("NRPy infrastructure:", par.parval_from_str("Infrastructure"))
for gridfunction_name in ["fdpg_phi", "fdpg_lap"]:
    gri.glb_gridfcs_dict.pop(gridfunction_name, None)
cfc.CFunction_dict.pop("fdpg_second_derivative", None)

fdpg_phi, fdpg_lap = gri.register_gridfunctions(
    ["fdpg_phi", "fdpg_lap"],
    group="AUX",
    dimension=1,
    f_infinity=[0.0, 0.0],
)
print("Registered gridfunctions:", fdpg_phi, fdpg_lap)

## Finite-Difference Coefficients

The stencil list stores three-dimensional offsets. For `dDD00`, only the first coordinate changes.

In [ ]:
fd_order = 4
coefficients, stencil = fin.compute_fdcoeffs_fdstencl("dDD00", fd_order=fd_order)

print("Derivative label: dDD00")
print("Finite-difference order:", fd_order)
print(" coefficient    stencil offset")
for coefficient, offset in zip(coefficients, stencil):
    print(f"{str(coefficient):>12}    {offset}")

## Small C Expression

`nrpy.c_codegen.c_codegen` converts a SymPy expression to C syntax. The expression below is the same second derivative written in terms of symbolic point samples.

In [ ]:
dx = sp.symbols("dx", positive=True)

sample_symbols = []
for offset in stencil:
    point_offset = offset[0]
    if point_offset < 0:
        name = f"phi_im{abs(point_offset)}"
    elif point_offset > 0:
        name = f"phi_ip{point_offset}"
    else:
        name = "phi_i"
    sample_symbols.append(sp.Symbol(name, real=True))

second_derivative = sum(
    coefficient * sample for coefficient, sample in zip(coefficients, sample_symbols)
) / dx**2

print(ccg.c_codegen(second_derivative, "d2phi_dx2", include_braces=False, verbose=False))

## Temporary C Program

The C program evaluates the NRPy stencil on $\phi(x)=\sin(2\pi x)$ with periodic ghost zones and compares it to the exact second derivative. The generated file lives in a temporary workspace.

In [ ]:
def c_index(base, offset):
    if offset == 0:
        return base
    sign = "+" if offset > 0 else "-"
    return f"{base} {sign} {abs(offset)}"


kernel_terms = []
for coefficient, offset in zip(coefficients, stencil):
    kernel_terms.append(
        f"({float(coefficient):+.17e}) * phi[{c_index('i', offset[0])}]"
    )
kernel_expression = "\n        + ".join(kernel_terms)

kernel_function = f"""void fdpg_second_derivative(const int n, const double dx, const double *phi, double *d2phi) {{
    for (int i = 2; i < n + 2; ++i) {{
        d2phi[i] = (
        {kernel_expression}
        ) / (dx * dx);
    }}
}}"""

try:
    cfc.register_CFunction(
        includes=["math.h"],
        desc="Fourth-order centered second derivative for the finite-difference playground.",
        cfunc_type="void",
        name="fdpg_second_derivative",
        params="const int n, const double dx, const double *phi, double *d2phi",
        body="\n".join(kernel_function.splitlines()[1:-1]),
    )
except Exception as exc:
    print(f"C function registry demonstration did not complete: {type(exc).__name__}: {exc}")
else:
    registered = cfc.CFunction_dict["fdpg_second_derivative"].full_function
    print("\n".join(registered.splitlines()[:16]))

workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy_fd_playground_")
workspace_path = Path(workspace_manager.name)
source_path = workspace_path / "fdpg_second_derivative.c"
executable_path = workspace_path / "fdpg_second_derivative"

c_program = f"""
#include <math.h>
#include <stdio.h>
#include <stdlib.h>

{kernel_function}

int main(void) {{
    const int n = 128;
    const int ng = 2;
    const double pi = acos(-1.0);
    const double dx = 1.0 / (double)n;
    double *phi = calloc((size_t)(n + 2 * ng), sizeof(double));
    double *d2phi = calloc((size_t)(n + 2 * ng), sizeof(double));

    if (phi == NULL || d2phi == NULL) {{
        return 2;
    }}

    for (int i = 0; i < n; ++i) {{
        const double x = ((double)i + 0.5) * dx;
        phi[i + ng] = sin(2.0 * pi * x);
    }}
    for (int g = 0; g < ng; ++g) {{
        phi[g] = phi[n + g];
        phi[n + ng + g] = phi[ng + g];
    }}

    fdpg_second_derivative(n, dx, phi, d2phi);

    double max_error = 0.0;
    for (int i = 0; i < n; ++i) {{
        const double x = ((double)i + 0.5) * dx;
        const double exact = -4.0 * pi * pi * sin(2.0 * pi * x);
        const double error = fabs(d2phi[i + ng] - exact);
        if (error > max_error) {{
            max_error = error;
        }}
    }}

    printf("n=%d dx=%.17e max_error=%.17e\\n", n, dx, max_error);
    free(phi);
    free(d2phi);
    return 0;
}}
"""

source_path.write_text(c_program, encoding="utf-8")
print("Temporary C source written:", source_path.name)

## Build and Run

The build step is optional. If `make` is absent but a C compiler is present, this notebook still compiles the single source file directly.

In [ ]:
import shutil


compiler = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
if compiler is None:
    print("C compiler unavailable; build skipped.")
else:
    compile_result = subprocess.run(
        [compiler, "-O2", source_path.name, "-lm", "-o", executable_path.name],
        cwd=workspace_path,
        text=True,
        capture_output=True,
        timeout=120,
    )
    print("Compile return code:", compile_result.returncode)
    if compile_result.stdout.strip():
        print(compile_result.stdout)
    if compile_result.returncode != 0:
        print("Compile stderr excerpt:")
        print("\n".join(compile_result.stderr.splitlines()[:12]))
        raise RuntimeError("Finite-difference playground C compilation failed.")
    else:
        run_result = subprocess.run(
            [f"./{executable_path.name}"],
            cwd=workspace_path,
            text=True,
            capture_output=True,
            timeout=120,
        )
        print("Run return code:", run_result.returncode)
        print(run_result.stdout.strip())
        if run_result.stderr.strip():
            print(run_result.stderr.strip())
        if run_result.returncode != 0:
            raise RuntimeError("Finite-difference playground executable failed.")

## Next Notebooks

The generated Cartesian wave notebooks use the same ingredients at project scale: symbolic right-hand sides, finite-difference stencils, C code generation, and Method of Lines integration. Continue with [Wave equation and C code generation](../3-wave_equation/wave_equation_and_c_codegen.ipynb) and [Boundary conditions and convergence](boundary_conditions_and_convergence.ipynb).